# 3. Pemodelan Latent Profile Analysis (LPA via GMM)
Tahap ini bertugas melakukan standardisasi fitur, evaluasi jumlah profil optimal (k) dengan BIC & Entropy, serta fitting model GMM dan visualisasi 2D PCA.

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Library Machine Learning berhasil di-import!')

In [ ]:
# Load dataset hasil preprocessing
file_path = None
for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    candidate = p / 'data' / 'processed' / 'student_features_clean.csv'
    if candidate.exists():
        file_path = candidate
        break

if not file_path:
    raise FileNotFoundError('student_features_clean.csv tidak ditemukan!')

df = pd.read_csv(file_path)
features = ['wellbeing', 'academic_pressure', 'social_support', 'career_readiness']
print(f'Data siap untuk pemodelan: {len(df)} responden')
df[features].head(3)

In [ ]:
# 1. Standardisasi Fitur (Z-Score)
X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Mean fitur setelah scaling (mendekati 0):', X_scaled.mean(axis=0).round(2))
print('Standar deviasi setelah scaling (tepat 1):', X_scaled.std(axis=0).round(2))

In [ ]:
# Fungsi perhitungan Entropy
def calc_entropy(gmm, X_sc):
    probs = np.clip(gmm.predict_proba(X_sc), 1e-12, 1.0)
    n_s, n_c = probs.shape
    ent = -np.sum(probs * np.log(probs))
    max_ent = n_s * np.log(n_c)
    return float(1.0 - (ent / max_ent))

# 2. Evaluasi Model Selection (k = 2 s.d. 5) dengan BIC, AIC, dan Entropy
eval_results = []
models = {}

for k in range(2, 6):
    gmm = GaussianMixture(n_components=k, covariance_type='diag', random_state=42, n_init=10)
    gmm.fit(X_scaled)
    bic_val = gmm.bic(X_scaled)
    aic_val = gmm.aic(X_scaled)
    ll_val = gmm.score(X_scaled) * len(X_scaled)
    ent_val = calc_entropy(gmm, X_scaled)
    
    models[k] = gmm
    eval_results.append({'k': k, 'BIC': bic_val, 'AIC': aic_val, 'Log-Likelihood': ll_val, 'Entropy': ent_val})

df_eval = pd.DataFrame(eval_results)
print('=== HASIL EVALUASI MODEL SELEKSI LPA (GMM) ===')
df_eval.round(2)

In [ ]:
# Visualisasi Kurva BIC dan Entropy
fig, ax1 = plt.subplots(figsize=(8, 4.5))

ax1.plot(df_eval['k'], df_eval['BIC'], marker='o', color='#047857', linewidth=2, label='BIC (Optimal Terendah)')
ax1.set_xlabel('Jumlah Profil Laten (k)')
ax1.set_ylabel('BIC', color='#047857')
ax1.tick_params(axis='y', labelcolor='#047857')

ax2 = ax1.twinx()
ax2.plot(df_eval['k'], df_eval['Entropy'], marker='s', color='#2563EB', linestyle='--', linewidth=2, label='Entropy')
ax2.set_ylabel('Entropy (Ketajaman Pemisahan)', color='#2563EB')
ax2.tick_params(axis='y', labelcolor='#2563EB')

plt.title('Kurva Seleksi Model LPA: BIC dan Entropy per Nilai k', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Fitting Model Terbaik (k = 4)
best_k = 4
gmm_chosen = models[best_k]
df['cluster'] = gmm_chosen.predict(X_scaled) + 1

# Probabilitas keanggotaan posterior
probs = gmm_chosen.predict_proba(X_scaled)
for c in range(best_k):
    df[f'prob_cluster_{c+1}'] = probs[:, c].round(3)

df[['student_id', 'prodi', 'cluster', 'prob_cluster_1', 'prob_cluster_2', 'prob_cluster_3', 'prob_cluster_4']].head()

In [ ]:
# 4. Rangkuman Karakteristik 4 Profil Laten Mahasiswa
profil_summary = df.groupby('cluster')[features].mean().round(1)
profil_summary['jumlah_mahasiswa'] = df['cluster'].value_counts()
profil_summary['persentase'] = (profil_summary['jumlah_mahasiswa'] / len(df) * 100).round(1)

print('=== KARAKTERISTIK 4 PROFIL LATEN MAHASISWA ===')
profil_summary

In [ ]:
# Visualisasi Karakteristik Skor tiap Profil
profil_summary[features].plot(kind='bar', figsize=(10, 5), colormap='viridis', width=0.75)
plt.title('Karakteristik Skor Rata-rata Tiap Profil Laten Mahasiswa', fontsize=13, fontweight='bold')
plt.xlabel('Profil Laten (Cluster)')
plt.ylabel('Skor Rata-rata (0 - 100)')
plt.xticks(ticks=range(4), labels=[f'Profil {i+1}' for i in range(4)], rotation=0)
plt.legend([f.replace('_', ' ').title() for f in features], bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Reduksi Dimensi 2D PCA
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df['pca_x'] = pca_coords[:, 0].round(2)
df['pca_y'] = pca_coords[:, 1].round(2)

print(f'Variansi PCA 1: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'Variansi PCA 2: {pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total variansi kumulatif: {pca.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# 6. Visualisasi Scatter Plot 2D PCA Sebaran Mahasiswa & Centroid
plt.figure(figsize=(9, 6))
palette = ['#E0664A', '#D9971E', '#2B65B0', '#059669']

for c in [1, 2, 3, 4]:
    sub = df[df['cluster'] == c]
    plt.scatter(sub['pca_x'], sub['pca_y'], color=palette[c-1], label=f'Profil {c} (n={len(sub)})', s=70, alpha=0.8, edgecolors='white')

# Centroid tiap klaster
for c in [1, 2, 3, 4]:
    sub = df[df['cluster'] == c]
    plt.scatter(sub['pca_x'].mean(), sub['pca_y'].mean(), color=palette[c-1], marker='X', s=280, edgecolors='black', linewidth=1.5)

plt.title('Visualisasi Sebaran Responden Mahasiswa PENS (LPA via GMM & PCA 2D)', fontsize=13, fontweight='bold')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Simpan Hasil Dataset & Parameter Model untuk Dashboard
out_dir = Path('../data/processed') if Path('../data').exists() else Path('data/processed')
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / 'student_features_clean.csv', index=False)

model_export = []
for c in [1, 2, 3, 4]:
    sub = df[df['cluster'] == c]
    means = sub[features].mean().round(1).to_dict()
    pca_c = {'x': float(sub['pca_x'].mean().round(2)), 'y': float(sub['pca_y'].mean().round(2))}
    model_export.append({
        'cluster_id': c,
        'percentage': round(float(len(sub) / len(df) * 100), 1),
        'n_students': int(len(sub)),
        'pca_coordinates': pca_c,
        'scores': means
    })

models_dir = Path('../models') if Path('../models').exists() else Path('models')
models_dir.mkdir(parents=True, exist_ok=True)
with open(models_dir / 'centroids.json', 'w', encoding='utf-8') as f:
    json.dump({'algorithm': 'LPA (GMM)', 'k': 4, 'clusters': model_export}, f, indent=2)

print('Data bersih dan centroids.json berhasil disimpan!')